# Enhanced LNN Model Training with Physics-Informed Features

This notebook trains an improved Liquid Neural Network (LNN) model using the enhanced dataset with molecular weight-derived physics-informed features. The goal is to significantly improve prediction accuracy and reduce overfitting compared to the baseline model.

## Key Enhancements:
1. **Physics-Informed Features** - Molecular weight-derived combustion parameters
2. **Advanced Architecture** - Attention mechanisms, residual connections
3. **Improved Regularization** - Advanced dropout, batch normalization
4. **Better Training Strategy** - AdamW optimizer, cosine annealing, gradient clipping
5. **Enhanced Data Quality** - Cleaned dataset with 9 input features vs. 4 original

## Expected Improvements:
- **Reduced Overfitting** - Physics-informed features provide better generalization
- **Higher R² Scores** - Real combustion physics vs. empirical correlations
- **Better ExcessO2 Prediction** - Model understands stoichiometric requirements
- **Improved OutletTemp Accuracy** - Actual heat input calculations

## Training Objectives:
1. Train enhanced LNN on physics-informed dataset
2. Compare performance against baseline model
3. Achieve positive R² scores and reduce overfitting
4. Demonstrate industrial applicability

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pickle
import os
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for publication quality
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 12

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("🚀 ENHANCED LNN TRAINING ENVIRONMENT")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"✅ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"   - CUDA version: {torch.version.cuda}")
    print(f"   - Device: {device}")
else:
    print("⚠️  Using CPU (GPU not available)")
    print(f"   - Device: {device}")

print(f"Random seeds set for reproducibility")
print("🎯 Ready for enhanced LNN training with physics-informed features!")
print("=" * 60)

🚀 ENHANCED LNN TRAINING ENVIRONMENT
PyTorch version: 2.2.0
NumPy version: 1.26.4
Pandas version: 2.2.2
⚠️  Using CPU (GPU not available)
   - Device: cpu
Random seeds set for reproducibility
🎯 Ready for enhanced LNN training with physics-informed features!


## 1. Load Enhanced Dataset with Physics-Informed Features

Load the enhanced dataset that includes molecular weight-derived features and verify the improvements over the baseline dataset.

In [2]:
# Load Enhanced Dataset with Physics-Informed Features
print("📊 LOADING ENHANCED DATASET")
print("=" * 50)

# Load the enhanced training data
enhanced_data_path = '/Users/abuhuzaifahbidin/Documents/GitHub/furnace-commander/backend/data/enhanced_training_data.pkl'

try:
    with open(enhanced_data_path, 'rb') as f:
        enhanced_data = pickle.load(f)
    
    # Extract enhanced data components
    X_train = enhanced_data['X_train']
    y_train = enhanced_data['y_train']
    X_val = enhanced_data['X_val']
    y_val = enhanced_data['y_val']
    X_test = enhanced_data['X_test']
    y_test = enhanced_data['y_test']
    
    input_features = enhanced_data['input_features']
    output_features = enhanced_data['output_features']
    normalization_params = enhanced_data['normalization_params']
    data_summary = enhanced_data['data_summary']
    
    print("✅ Enhanced dataset loaded successfully!")
    
except FileNotFoundError:
    print(f"❌ Enhanced dataset not found: {enhanced_data_path}")
    print("Please run the benchmark_data2_analysis.ipynb notebook first")
    raise

print(f"\n📈 ENHANCED DATASET SUMMARY")
print("=" * 50)
print(f"Training samples:   {X_train.shape[0]:,}")
print(f"Validation samples: {X_val.shape[0]:,}")
print(f"Test samples:       {X_test.shape[0]:,}")
print(f"Total samples:      {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]:,}")

print(f"\n🔧 ENHANCED DATA STRUCTURE")
print("=" * 50)
print(f"Input shape:  {X_train.shape} (samples, timesteps, features)")
print(f"Output shape: {y_train.shape} (samples, targets)")
print(f"Sequence length: {X_train.shape[1]} timesteps")

print(f"\n🚀 PHYSICS-INFORMED INPUT FEATURES ({len(input_features)}):")
print("=" * 60)
for i, feature in enumerate(input_features, 1):
    feature_type = "🔬 Physics" if feature in ['FuelDensity_std', 'SpecificGravity', 'WobbeIndex_approx', 
                                              'StoichAFR', 'FuelMassFlow', 'ActualHeatInput', 
                                              'ExcessAirRatio', 'CombustionEfficiency'] else "⚙️  Process"
    print(f"{i:2d}. {feature_type} {feature}")

print(f"\n🎯 OUTPUT TARGETS ({len(output_features)}):")
print("=" * 50)
for i, feature in enumerate(output_features, 1):
    print(f"{i:2d}. {feature}")

print(f"\n✅ DATA QUALITY CHECK")
print("=" * 50)
print(f"X_train range: [{X_train.min():.6f}, {X_train.max():.6f}]")
print(f"y_train range: [{y_train.min():.6f}, {y_train.max():.6f}]")
print(f"No NaN values: {not (np.isnan(X_train).any() or np.isnan(y_train).any())}")
print(f"No infinite values: {not (np.isinf(X_train).any() or np.isinf(y_train).any())}")

# Compare with baseline dataset
baseline_features = ['InletTemp', 'InletFlow', 'AFR', 'FuelFlow']
physics_features = [f for f in input_features if f not in baseline_features]

print(f"\n📊 ENHANCEMENT COMPARISON")
print("=" * 50)
print(f"Baseline features: {len(baseline_features)} → Enhanced features: {len(input_features)}")
print(f"Added physics features: {len(physics_features)}")
print(f"Feature enhancement: +{len(physics_features)/len(baseline_features)*100:.0f}%")

print(f"\n🔬 PHYSICS-INFORMED FEATURES ADDED:")
for i, feature in enumerate(physics_features, 1):
    print(f"{i:2d}. {feature}")

print(f"\nEnhanced dataset ready for training! 🚀")

📊 LOADING ENHANCED DATASET
✅ Enhanced dataset loaded successfully!

📈 ENHANCED DATASET SUMMARY
Training samples:   115,963
Validation samples: 24,849
Test samples:       24,850
Total samples:      165,662

🔧 ENHANCED DATA STRUCTURE
Input shape:  (115963, 10, 13) (samples, timesteps, features)
Output shape: (115963, 2) (samples, targets)
Sequence length: 10 timesteps

🚀 PHYSICS-INFORMED INPUT FEATURES (13):
 1. ⚙️  Process InletTemp
 2. ⚙️  Process InletFlow
 3. ⚙️  Process AirFuelRatio
 4. ⚙️  Process FuelGasFlow
 5. 🔬 Physics FuelDensity_std
 6. 🔬 Physics SpecificGravity
 7. 🔬 Physics WobbeIndex_approx
 8. 🔬 Physics StoichAFR
 9. ⚙️  Process FuelGasMW
10. 🔬 Physics FuelMassFlow
11. 🔬 Physics ActualHeatInput
12. 🔬 Physics ExcessAirRatio
13. 🔬 Physics CombustionEfficiency

🎯 OUTPUT TARGETS (2):
 1. OutletTemp
 2. ExcessO2

✅ DATA QUALITY CHECK
X_train range: [0.000000, 1.000000]
y_train range: [0.000000, 1.000000]
No NaN values: True
No infinite values: True

📊 ENHANCEMENT COMPARISON
Ba

## 2. Enhanced LNN Architecture Design

Design an improved Liquid Neural Network architecture optimized for physics-informed features with advanced regularization and attention mechanisms.

In [3]:
class EnhancedLNNFurnaceModel(nn.Module):
    """
    Enhanced Liquid Neural Network for furnace control with physics-informed features.
    
    Key improvements:
    - Attention mechanisms for feature importance
    - Residual connections to prevent gradient vanishing
    - Advanced regularization (dropout, batch normalization)
    - Specialized heads for different outputs
    - Physics-aware architecture design
    """
    
    def __init__(self, sequence_length=10, n_features=9, n_outputs=2, dropout_rate=0.3):
        super(EnhancedLNNFurnaceModel, self).__init__()
        
        self.sequence_length = sequence_length
        self.n_features = n_features
        self.n_outputs = n_outputs
        self.dropout_rate = dropout_rate
        
        # Input embedding for physics-informed features
        self.feature_embedding = nn.Linear(n_features, 64)
        self.feature_norm = nn.LayerNorm(64)
        
        # Enhanced LSTM layers with bidirectional processing
        self.lstm1 = nn.LSTM(64, 128, batch_first=True, dropout=dropout_rate, bidirectional=True)
        self.lstm1_norm = nn.LayerNorm(256)  # 128 * 2 for bidirectional
        
        self.lstm2 = nn.LSTM(256, 96, batch_first=True, dropout=dropout_rate, bidirectional=True)
        self.lstm2_norm = nn.LayerNorm(192)  # 96 * 2 for bidirectional
        
        # Multi-head attention for temporal and feature relationships
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=192, num_heads=8, dropout=dropout_rate, batch_first=True
        )
        self.attention_norm = nn.LayerNorm(192)
        
        # Feature importance gating
        self.feature_gate = nn.Sequential(
            nn.Linear(192, 96),
            nn.ReLU(),
            nn.Linear(96, 192),
            nn.Sigmoid()
        )
        
        # Enhanced dense processing with residual connections
        self.dense1 = nn.Linear(192, 256)
        self.dense1_norm = nn.LayerNorm(256)
        self.dense1_dropout = nn.Dropout(dropout_rate)
        
        self.dense2 = nn.Linear(256, 128)
        self.dense2_norm = nn.LayerNorm(128)
        self.dense2_dropout = nn.Dropout(dropout_rate * 0.8)
        
        self.dense3 = nn.Linear(128, 64)
        self.dense3_norm = nn.LayerNorm(64)
        self.dense3_dropout = nn.Dropout(dropout_rate * 0.6)
        
        # Specialized output heads for physics-aware predictions
        # Temperature head - focuses on thermal dynamics
        self.temp_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.4),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
        # Excess O2 head - focuses on combustion efficiency
        self.o2_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.4),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
        # Activation functions
        self.relu = nn.ReLU()
        self.gelu = nn.GELU()
        self.sigmoid = nn.Sigmoid()
        
        # Initialize weights
        self._initialize_weights()
        
    def _initialize_weights(self):
        """Initialize weights using Xavier/He initialization for better convergence."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LSTM):
                for name, param in module.named_parameters():
                    if 'weight' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'bias' in name:
                        nn.init.zeros_(param)
        
    def forward(self, x):
        batch_size = x.size(0)
        
        # Feature embedding for physics-informed inputs
        x = self.feature_embedding(x)
        x = self.feature_norm(x)
        x = self.gelu(x)
        
        # Enhanced LSTM processing with bidirectional information flow
        lstm1_out, _ = self.lstm1(x)
        lstm1_out = self.lstm1_norm(lstm1_out)
        
        lstm2_out, _ = self.lstm2(lstm1_out)
        lstm2_out = self.lstm2_norm(lstm2_out)
        
        # Residual connection if dimensions match
        if lstm1_out.size(-1) == lstm2_out.size(-1):
            lstm2_out = lstm2_out + lstm1_out
        
        # Multi-head attention for capturing complex relationships
        attn_out, attn_weights = self.temporal_attention(lstm2_out, lstm2_out, lstm2_out)
        attn_out = self.attention_norm(attn_out + lstm2_out)  # Residual connection
        
        # Feature importance gating
        feature_importance = self.feature_gate(attn_out)
        gated_features = attn_out * feature_importance
        
        # Global pooling to combine temporal information
        pooled = torch.mean(gated_features, dim=1)
        
        # Enhanced dense processing with residual connections
        x = self.dense1(pooled)
        x = self.dense1_norm(x)
        x = self.gelu(x)
        x = self.dense1_dropout(x)
        
        dense1_out = x
        
        x = self.dense2(x)
        x = self.dense2_norm(x)
        x = self.gelu(x)
        x = self.dense2_dropout(x)
        
        x = self.dense3(x)
        x = self.dense3_norm(x)
        x = self.gelu(x)
        x = self.dense3_dropout(x)
        
        # Specialized predictions
        temp_pred = self.temp_head(x)
        o2_pred = self.o2_head(x)
        
        # Combine outputs
        output = torch.cat([temp_pred, o2_pred], dim=1)
        output = self.sigmoid(output)  # Normalize to [0,1] for normalized targets
        
        return output

# Create the enhanced model
print("🏗️  BUILDING ENHANCED LNN MODEL")
print("=" * 50)

enhanced_model = EnhancedLNNFurnaceModel(
    sequence_length=X_train.shape[1],
    n_features=X_train.shape[2],
    n_outputs=y_train.shape[1],
    dropout_rate=0.4  # Increased dropout to combat overfitting
).to(device)

# Count parameters
total_params = sum(p.numel() for p in enhanced_model.parameters())
trainable_params = sum(p.numel() for p in enhanced_model.parameters() if p.requires_grad)

print(f"✅ Enhanced LNN model created successfully!")

print(f"\n📊 MODEL STATISTICS")
print("=" * 50)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model memory usage: ~{total_params * 4 / 1024**2:.2f} MB (float32)")
print(f"Device: {device}")

print(f"\n🎯 ENHANCED ARCHITECTURE FEATURES")
print("=" * 60)
print("✨ Physics-informed feature embedding")
print("✨ Bidirectional LSTM for temporal modeling")
print("✨ Multi-head attention mechanism")
print("✨ Feature importance gating")
print("✨ Residual connections")
print("✨ Layer normalization for stability")
print("✨ Specialized output heads (Temperature + O2)")
print("✨ Advanced regularization (40% dropout)")
print("✨ GELU activations for smooth gradients")
print("✨ Xavier weight initialization")

print(f"\n🔬 PHYSICS-AWARE DESIGN")
print("=" * 50)
print("• Separate heads for thermal vs. combustion dynamics")
print("• Feature gating for physics parameter importance")
print("• Architecture sized for enhanced feature complexity")
print("• Regularization tuned for physics-informed data")

print(f"\nReady for enhanced training! 🚀")

🏗️  BUILDING ENHANCED LNN MODEL
✅ Enhanced LNN model created successfully!

📊 MODEL STATISTICS
Total parameters: 754,914
Trainable parameters: 754,914
Model memory usage: ~2.88 MB (float32)
Device: cpu

🎯 ENHANCED ARCHITECTURE FEATURES
✨ Physics-informed feature embedding
✨ Bidirectional LSTM for temporal modeling
✨ Multi-head attention mechanism
✨ Feature importance gating
✨ Residual connections
✨ Layer normalization for stability
✨ Specialized output heads (Temperature + O2)
✨ Advanced regularization (40% dropout)
✨ GELU activations for smooth gradients
✨ Xavier weight initialization

🔬 PHYSICS-AWARE DESIGN
• Separate heads for thermal vs. combustion dynamics
• Feature gating for physics parameter importance
• Architecture sized for enhanced feature complexity
• Regularization tuned for physics-informed data

Ready for enhanced training! 🚀


## 3. Advanced Training Configuration

Set up advanced training configuration with physics-informed loss functions and state-of-the-art optimization strategies.

In [4]:
# Advanced Training Configuration for Enhanced Model
print("⚙️  ADVANCED TRAINING CONFIGURATION")
print("=" * 60)

# Enhanced hyperparameters optimized for physics-informed features
BATCH_SIZE = 32          # Smaller batch for better gradient estimates with complex model
LEARNING_RATE = 0.0003   # Conservative learning rate for stable training
NUM_EPOCHS = 200         # More epochs for complex model convergence
PATIENCE = 35            # Increased patience for complex model

print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Max epochs: {NUM_EPOCHS}")
print(f"Early stopping patience: {PATIENCE}")

# Convert data to PyTorch tensors
print(f"\n🔄 PREPARING ENHANCED DATA FOR TRAINING")
print("=" * 60)

X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.FloatTensor(y_train).to(device)
X_val_tensor = torch.FloatTensor(X_val).to(device)
y_val_tensor = torch.FloatTensor(y_val).to(device)
X_test_tensor = torch.FloatTensor(X_test).to(device)
y_test_tensor = torch.FloatTensor(y_test).to(device)

print(f"✅ Training data: {X_train_tensor.shape} → {y_train_tensor.shape}")
print(f"✅ Validation data: {X_val_tensor.shape} → {y_val_tensor.shape}")
print(f"✅ Test data: {X_test_tensor.shape} → {y_test_tensor.shape}")

# Create enhanced DataLoaders with optimized settings
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    pin_memory=True,
    num_workers=0  # Set to 0 for compatibility
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    pin_memory=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    pin_memory=True,
    num_workers=0
)

print(f"✅ Enhanced DataLoaders created:")
print(f"   - Training batches: {len(train_loader)}")
print(f"   - Validation batches: {len(val_loader)}")
print(f"   - Test batches: {len(test_loader)}")

# Physics-Informed Loss Function
class PhysicsInformedLoss(nn.Module):
    """
    Physics-informed loss function that balances temperature and combustion predictions
    with awareness of the underlying physics.
    """
    def __init__(self, temp_weight=1.0, o2_weight=1.0, physics_penalty=0.1):
        super(PhysicsInformedLoss, self).__init__()
        self.temp_weight = temp_weight
        self.o2_weight = o2_weight
        self.physics_penalty = physics_penalty
        
    def forward(self, predictions, targets):
        # Separate temperature and O2 predictions
        temp_pred = predictions[:, 0]
        o2_pred = predictions[:, 1]
        temp_target = targets[:, 0]
        o2_target = targets[:, 1]
        
        # Individual MSE losses
        temp_loss = torch.mean((temp_pred - temp_target) ** 2)
        o2_loss = torch.mean((o2_pred - o2_target) ** 2)
        
        # Weighted combination
        weighted_loss = self.temp_weight * temp_loss + self.o2_weight * o2_loss
        
        # Physics penalty: Discourage unphysical predictions
        # (e.g., very high O2 with very high temperatures)
        physics_violation = torch.mean(torch.relu(temp_pred * o2_pred - 0.8))  # Threshold tunable
        
        total_loss = weighted_loss + self.physics_penalty * physics_violation
        
        return total_loss

# Enhanced optimizer and scheduler
criterion = PhysicsInformedLoss(temp_weight=1.0, o2_weight=1.5, physics_penalty=0.05)

# AdamW optimizer with weight decay for better generalization
optimizer = optim.AdamW(
    enhanced_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
    betas=(0.9, 0.999),
    eps=1e-8
)

# Cosine annealing with warm restarts for optimal learning rate scheduling
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=25,      # Initial restart interval
    T_mult=2,    # Multiply restart interval by this factor
    eta_min=1e-6 # Minimum learning rate
)

print(f"\n🎯 ADVANCED TRAINING SETUP")
print("=" * 60)
print(f"Loss function: Physics-Informed MSE")
print(f"  - Temperature weight: {criterion.temp_weight}")
print(f"  - ExcessO2 weight: {criterion.o2_weight}")
print(f"  - Physics penalty: {criterion.physics_penalty}")
print(f"Optimizer: AdamW with weight decay (1e-4)")
print(f"Scheduler: CosineAnnealingWarmRestarts")
print(f"Gradient clipping: Enabled (max_norm=1.0)")
print(f"Mixed precision: {'Enabled' if device.type == 'cuda' else 'Disabled'}")

# Additional training utilities
def calculate_metrics(predictions, targets):
    """Calculate comprehensive metrics for model evaluation."""
    predictions_np = predictions.detach().cpu().numpy()
    targets_np = targets.detach().cpu().numpy()
    
    metrics = {}
    target_names = ['OutletTemp', 'ExcessO2']
    
    for i, target in enumerate(target_names):
        pred = predictions_np[:, i]
        actual = targets_np[:, i]
        
        metrics[target] = {
            'MSE': np.mean((pred - actual) ** 2),
            'MAE': np.mean(np.abs(pred - actual)),
            'RMSE': np.sqrt(np.mean((pred - actual) ** 2)),
            'R2': 1 - (np.sum((actual - pred) ** 2) / np.sum((actual - np.mean(actual)) ** 2))
        }
    
    return metrics

print(f"\n🚀 Enhanced training configuration ready!")
print(f"Ready to train physics-informed LNN model!")
print("=" * 60)

⚙️  ADVANCED TRAINING CONFIGURATION
Batch size: 32
Learning rate: 0.0003
Max epochs: 200
Early stopping patience: 35

🔄 PREPARING ENHANCED DATA FOR TRAINING
✅ Training data: torch.Size([115963, 10, 13]) → torch.Size([115963, 2])
✅ Validation data: torch.Size([24849, 10, 13]) → torch.Size([24849, 2])
✅ Test data: torch.Size([24850, 10, 13]) → torch.Size([24850, 2])
✅ Enhanced DataLoaders created:
   - Training batches: 3624
   - Validation batches: 777
   - Test batches: 777

🎯 ADVANCED TRAINING SETUP
Loss function: Physics-Informed MSE
  - Temperature weight: 1.0
  - ExcessO2 weight: 1.5
  - Physics penalty: 0.05
Optimizer: AdamW with weight decay (1e-4)
Scheduler: CosineAnnealingWarmRestarts
Gradient clipping: Enabled (max_norm=1.0)
Mixed precision: Disabled

🚀 Enhanced training configuration ready!
Ready to train physics-informed LNN model!


## 4. Enhanced Model Training

Train the enhanced LNN model with physics-informed features using advanced techniques including gradient clipping, learning rate scheduling, and comprehensive monitoring.

In [5]:
# Enhanced Model Training with Advanced Techniques
def train_enhanced_model(model, train_loader, val_loader, criterion, optimizer, scheduler, 
                        num_epochs, patience, device):
    """
    Train the enhanced LNN model with advanced techniques:
    - Physics-informed loss function
    - Gradient clipping
    - Advanced early stopping
    - Comprehensive metric tracking
    - Mixed precision (if available)
    """
    
    # Training history with enhanced metrics
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_mae': [],
        'val_mae': [],
        'train_r2_temp': [],
        'train_r2_o2': [],
        'val_r2_temp': [],
        'val_r2_o2': [],
        'learning_rates': [],
        'grad_norms': []
    }
    
    # Advanced early stopping
    best_val_loss = float('inf')
    best_val_r2 = -float('inf')
    patience_counter = 0
    best_model_state = None
    
    # Mixed precision scaler (if CUDA available)
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None
    
    print("🚀 STARTING ENHANCED LNN TRAINING")
    print("=" * 80)
    print(f"{'Epoch':<6} {'Train Loss':<12} {'Val Loss':<12} {'Train R²':<12} {'Val R²':<12} {'LR':<10} {'Grad':<8}")
    print("=" * 80)
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_mae = 0.0
        train_metrics_accumulator = {'OutletTemp': {'r2_num': 0, 'r2_den': 0}, 
                                   'ExcessO2': {'r2_num': 0, 'r2_den': 0}}
        num_train_batches = 0
        total_grad_norm = 0.0
        
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            
            # Mixed precision forward pass
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = model(batch_x)
                    loss = criterion(outputs, batch_y)
                
                # Mixed precision backward pass
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                # Standard training
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                
                # Gradient clipping
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            
            # Accumulate metrics
            train_loss += loss.item()
            train_mae += torch.mean(torch.abs(outputs - batch_y)).item()
            total_grad_norm += grad_norm.item()
            
            # Accumulate R² components
            for i, target in enumerate(['OutletTemp', 'ExcessO2']):
                pred = outputs[:, i]
                actual = batch_y[:, i]
                train_metrics_accumulator[target]['r2_num'] += torch.sum((actual - pred) ** 2).item()
                train_metrics_accumulator[target]['r2_den'] += torch.sum((actual - torch.mean(actual)) ** 2).item()
            
            num_train_batches += 1
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_mae = 0.0
        val_metrics_accumulator = {'OutletTemp': {'r2_num': 0, 'r2_den': 0}, 
                                 'ExcessO2': {'r2_num': 0, 'r2_den': 0}}
        num_val_batches = 0
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                if scaler is not None:
                    with torch.cuda.amp.autocast():
                        outputs = model(batch_x)
                        loss = criterion(outputs, batch_y)
                else:
                    outputs = model(batch_x)
                    loss = criterion(outputs, batch_y)
                
                val_loss += loss.item()
                val_mae += torch.mean(torch.abs(outputs - batch_y)).item()
                
                # Accumulate R² components
                for i, target in enumerate(['OutletTemp', 'ExcessO2']):
                    pred = outputs[:, i]
                    actual = batch_y[:, i]
                    val_metrics_accumulator[target]['r2_num'] += torch.sum((actual - pred) ** 2).item()
                    val_metrics_accumulator[target]['r2_den'] += torch.sum((actual - torch.mean(actual)) ** 2).item()
                
                num_val_batches += 1
        
        # Calculate average metrics
        avg_train_loss = train_loss / num_train_batches
        avg_val_loss = val_loss / num_val_batches
        avg_train_mae = train_mae / num_train_batches
        avg_val_mae = val_mae / num_val_batches
        avg_grad_norm = total_grad_norm / num_train_batches
        
        # Calculate R² scores
        train_r2_temp = 1 - (train_metrics_accumulator['OutletTemp']['r2_num'] / 
                           train_metrics_accumulator['OutletTemp']['r2_den'])
        train_r2_o2 = 1 - (train_metrics_accumulator['ExcessO2']['r2_num'] / 
                         train_metrics_accumulator['ExcessO2']['r2_den'])
        val_r2_temp = 1 - (val_metrics_accumulator['OutletTemp']['r2_num'] / 
                         val_metrics_accumulator['OutletTemp']['r2_den'])
        val_r2_o2 = 1 - (val_metrics_accumulator['ExcessO2']['r2_num'] / 
                       val_metrics_accumulator['ExcessO2']['r2_den'])
        
        avg_train_r2 = (train_r2_temp + train_r2_o2) / 2
        avg_val_r2 = (val_r2_temp + val_r2_o2) / 2
        
        # Store history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_mae'].append(avg_train_mae)
        history['val_mae'].append(avg_val_mae)
        history['train_r2_temp'].append(train_r2_temp)
        history['train_r2_o2'].append(train_r2_o2)
        history['val_r2_temp'].append(val_r2_temp)
        history['val_r2_o2'].append(val_r2_o2)
        history['learning_rates'].append(optimizer.param_groups[0]['lr'])
        history['grad_norms'].append(avg_grad_norm)
        
        # Learning rate scheduling
        scheduler.step()
        
        # Print progress
        if epoch % 5 == 0 or epoch < 10:
            print(f"{epoch+1:<6} {avg_train_loss:<12.6f} {avg_val_loss:<12.6f} "
                  f"{avg_train_r2:<12.4f} {avg_val_r2:<12.4f} {optimizer.param_groups[0]['lr']:<10.2e} {avg_grad_norm:<8.4f}")
        
        # Enhanced early stopping based on validation R²
        improvement_threshold = 1e-6
        if avg_val_r2 > best_val_r2 + improvement_threshold:
            best_val_r2 = avg_val_r2
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered after epoch {epoch+1}")
            print(f"Best validation R²: {best_val_r2:.6f}")
            print(f"Best validation loss: {best_val_loss:.6f}")
            break
    
    # Load best model state
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    training_time = time.time() - start_time
    
    print("=" * 80)
    print(f"Enhanced training completed in {training_time:.2f} seconds")
    print(f"Best validation R²: {best_val_r2:.6f}")
    print(f"Best validation loss: {best_val_loss:.6f}")
    print(f"Final learning rate: {optimizer.param_groups[0]['lr']:.2e}")
    print(f"Average gradient norm: {np.mean(history['grad_norms']):.4f}")
    
    return model, history

# Start enhanced training
print("🎯 TRAINING ENHANCED LNN WITH PHYSICS-INFORMED FEATURES")
print("=" * 70)

enhanced_trained_model, enhanced_history = train_enhanced_model(
    model=enhanced_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    device=device
)

print("\n✅ Enhanced model training completed!")
print("🔬 Physics-informed LNN ready for evaluation and benchmarking!")
print("=" * 70)

🎯 TRAINING ENHANCED LNN WITH PHYSICS-INFORMED FEATURES
🚀 STARTING ENHANCED LNN TRAINING
Epoch  Train Loss   Val Loss     Train R²     Val R²       LR         Grad    
1      0.002995     0.012283     0.0738       -111.5232    2.99e-04   0.0428  
2      0.001052     0.008480     0.6711       -91.0341     2.95e-04   0.0279  
3      0.000723     0.008670     0.7774       -99.9280     2.90e-04   0.0237  
4      0.000600     0.008796     0.8173       -106.4047    2.82e-04   0.0215  
5      0.000556     0.007314     0.8306       -83.9306     2.71e-04   0.0221  
6      0.000486     0.008713     0.8541       -107.5676    2.59e-04   0.0199  
7      0.000478     0.008301     0.8562       -102.3381    2.46e-04   0.0212  
8      0.000441     0.008169     0.8684       -99.7780     2.31e-04   0.0198  
9      0.000445     0.008415     0.8662       -103.4043    2.14e-04   0.0199  
10     0.000406     0.008585     0.8791       -102.6434    1.97e-04   0.0187  


KeyboardInterrupt: 

## 5. Anti-Overfitting Strategies

Implement aggressive regularization and anti-overfitting techniques to improve generalization and reduce the training/validation gap.

In [10]:
# Anti-Overfitting Model with Aggressive Regularization
class RegularizedLNNModel(nn.Module):
    """
    LNN Model with aggressive anti-overfitting techniques:
    - Much higher dropout rates
    - Batch normalization after every layer
    - L2 regularization through weight decay
    - Smaller model capacity to prevent memorization
    - Residual connections with regularization
    """
    
    def __init__(self, sequence_length=10, n_features=9, n_outputs=2):
        super(RegularizedLNNModel, self).__init__()
        
        # Reduced model capacity to prevent overfitting
        self.feature_embedding = nn.Linear(n_features, 32)  # Reduced from 64
        self.embedding_dropout = nn.Dropout(0.2)
        self.embedding_bn = nn.BatchNorm1d(32)
        
        # Smaller LSTM layers with aggressive dropout
        self.lstm1 = nn.LSTM(32, 64, batch_first=True, dropout=0.5)  # High dropout
        self.lstm1_bn = nn.BatchNorm1d(64)
        self.lstm1_dropout = nn.Dropout(0.6)
        
        self.lstm2 = nn.LSTM(64, 32, batch_first=True, dropout=0.5)
        self.lstm2_bn = nn.BatchNorm1d(32)
        self.lstm2_dropout = nn.Dropout(0.6)
        
        # Simple attention with regularization
        self.attention = nn.MultiheadAttention(embed_dim=32, num_heads=4, dropout=0.5, batch_first=True)
        self.attention_dropout = nn.Dropout(0.6)
        self.attention_bn = nn.BatchNorm1d(32)
        
        # Smaller dense layers with heavy regularization
        self.dense1 = nn.Linear(32, 64)
        self.dense1_bn = nn.BatchNorm1d(64)
        self.dense1_dropout = nn.Dropout(0.7)  # Very high dropout
        
        self.dense2 = nn.Linear(64, 32)
        self.dense2_bn = nn.BatchNorm1d(32)
        self.dense2_dropout = nn.Dropout(0.6)
        
        # Simple output layer
        self.output = nn.Linear(32, n_outputs)
        self.output_dropout = nn.Dropout(0.3)
        
        # Activations
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
        # Initialize with smaller weights to prevent overfitting
        self._initialize_weights()
        
    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                # Smaller initial weights
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.LSTM):
                for name, param in module.named_parameters():
                    if 'weight' in name:
                        nn.init.normal_(param, mean=0.0, std=0.02)
                    elif 'bias' in name:
                        nn.init.zeros_(param)
    
    def forward(self, x):
        batch_size, seq_len, features = x.size()
        
        # Feature embedding with regularization
        x = self.feature_embedding(x)
        x = x.view(-1, x.size(-1))
        x = self.embedding_bn(x)
        x = x.view(batch_size, seq_len, -1)
        x = self.embedding_dropout(x)
        x = self.relu(x)
        
        # LSTM processing with heavy regularization
        lstm1_out, _ = self.lstm1(x)
        
        # Apply batch norm (reshape for batch norm)
        lstm1_out = lstm1_out.reshape(-1, lstm1_out.size(-1))
        lstm1_out = self.lstm1_bn(lstm1_out)
        lstm1_out = lstm1_out.reshape(batch_size, seq_len, -1)
        lstm1_out = self.lstm1_dropout(lstm1_out)
        
        lstm2_out, _ = self.lstm2(lstm1_out)
        lstm2_out = lstm2_out.reshape(-1, lstm2_out.size(-1))
        lstm2_out = self.lstm2_bn(lstm2_out)
        lstm2_out = lstm2_out.reshape(batch_size, seq_len, -1)
        lstm2_out = self.lstm2_dropout(lstm2_out)
        
        # Simple attention
        attn_out, _ = self.attention(lstm2_out, lstm2_out, lstm2_out)
        attn_out = attn_out.reshape(-1, attn_out.size(-1))
        attn_out = self.attention_bn(attn_out)
        attn_out = attn_out.reshape(batch_size, seq_len, -1)
        attn_out = self.attention_dropout(attn_out)
        
        # Global average pooling
        pooled = torch.mean(attn_out, dim=1)
        
        # Dense layers with heavy regularization
        x = self.dense1(pooled)
        x = self.dense1_bn(x)
        x = self.relu(x)
        x = self.dense1_dropout(x)
        
        x = self.dense2(x)
        x = self.dense2_bn(x)
        x = self.relu(x)
        x = self.dense2_dropout(x)
        
        # Output
        x = self.output_dropout(x)
        x = self.output(x)
        x = self.sigmoid(x)
        
        return x

# Data Augmentation for Time Series
class TimeSeriesAugmentation:
    """Data augmentation techniques for time series to increase dataset diversity."""
    
    def __init__(self, noise_std=0.01, scaling_range=(0.95, 1.05)):
        self.noise_std = noise_std
        self.scaling_range = scaling_range
    
    def add_noise(self, x):
        """Add Gaussian noise to input."""
        noise = torch.normal(0, self.noise_std, size=x.shape).to(x.device)
        return x + noise
    
    def scale_features(self, x):
        """Apply random scaling to features."""
        scale = torch.empty(x.shape[0], 1, x.shape[2]).uniform_(*self.scaling_range).to(x.device)
        return x * scale
    
    def temporal_shift(self, x, max_shift=2):
        """Apply small temporal shifts."""
        batch_size, seq_len, features = x.shape
        shifted_x = x.clone()
        
        for i in range(batch_size):
            shift = torch.randint(-max_shift, max_shift + 1, (1,)).item()
            if shift != 0:
                if shift > 0:
                    shifted_x[i, :-shift] = x[i, shift:]
                    shifted_x[i, -shift:] = x[i, -1:].repeat(shift, 1)
                else:
                    shifted_x[i, -shift:] = x[i, :shift]
                    shifted_x[i, :-shift] = x[i, 0:1].repeat(-shift, 1)
        
        return shifted_x

# Create regularized model
print("🛡️  BUILDING REGULARIZED ANTI-OVERFITTING MODEL")
print("=" * 60)

regularized_model = RegularizedLNNModel(
    sequence_length=X_train.shape[1],
    n_features=X_train.shape[2],
    n_outputs=y_train.shape[1]
).to(device)

# Count parameters
reg_total_params = sum(p.numel() for p in regularized_model.parameters())
original_total_params = sum(p.numel() for p in enhanced_model.parameters())

print(f"✅ Regularized model created!")
print(f"Original model parameters: {original_total_params:,}")
print(f"Regularized model parameters: {reg_total_params:,}")
print(f"Parameter reduction: {(1 - reg_total_params/original_total_params)*100:.1f}%")

# Enhanced training configuration with stronger regularization
REGULARIZED_BATCH_SIZE = 64      # Larger batch for better gradient estimates
REGULARIZED_LEARNING_RATE = 0.0001  # Lower learning rate
REGULARIZED_NUM_EPOCHS = 300
REGULARIZED_PATIENCE = 50         # More patience

# Create data augmentation
augmentation = TimeSeriesAugmentation(noise_std=0.005, scaling_range=(0.98, 1.02))

# Enhanced loss function with L2 regularization
class RegularizedLoss(nn.Module):
    def __init__(self, l2_lambda=0.01):
        super(RegularizedLoss, self).__init__()
        self.l2_lambda = l2_lambda
        self.mse = nn.MSELoss()
    
    def forward(self, predictions, targets, model):
        # Base MSE loss
        mse_loss = self.mse(predictions, targets)
        
        # L2 regularization
        l2_reg = 0
        for param in model.parameters():
            l2_reg += torch.norm(param, 2) ** 2
        
        total_loss = mse_loss + self.l2_lambda * l2_reg
        return total_loss

regularized_criterion = RegularizedLoss(l2_lambda=0.001)

# Conservative optimizer with higher weight decay
regularized_optimizer = optim.AdamW(
    regularized_model.parameters(),
    lr=REGULARIZED_LEARNING_RATE,
    weight_decay=0.01,  # Much higher weight decay
    betas=(0.9, 0.999)
)

# Learning rate scheduler with slower decay
regularized_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    regularized_optimizer,
    mode='min',
    factor=0.8,
    patience=15,
    min_lr=1e-7,
    verbose=True
)

print(f"\n🎯 ANTI-OVERFITTING STRATEGIES")
print("=" * 60)
print("✅ Reduced model capacity (50% fewer parameters)")
print("✅ Aggressive dropout (50-70% rates)")
print("✅ Batch normalization after every layer")
print("✅ L2 regularization (λ=0.001)")
print("✅ Higher weight decay (0.01)")
print("✅ Data augmentation (noise + scaling + temporal shifts)")
print("✅ Lower learning rate (0.0001)")
print("✅ Larger batch size (64)")
print("✅ Conservative learning rate scheduling")

print(f"\nReady for anti-overfitting training! 🛡️")

🛡️  BUILDING REGULARIZED ANTI-OVERFITTING MODEL
✅ Regularized model created!
Original model parameters: 754,914
Regularized model parameters: 47,074
Parameter reduction: 93.8%

🎯 ANTI-OVERFITTING STRATEGIES
✅ Reduced model capacity (50% fewer parameters)
✅ Aggressive dropout (50-70% rates)
✅ Batch normalization after every layer
✅ L2 regularization (λ=0.001)
✅ Higher weight decay (0.01)
✅ Data augmentation (noise + scaling + temporal shifts)
✅ Lower learning rate (0.0001)
✅ Larger batch size (64)
✅ Conservative learning rate scheduling

Ready for anti-overfitting training! 🛡️


In [11]:
# Train Regularized Model with Anti-Overfitting Techniques
def train_regularized_model(model, train_loader, val_loader, criterion, optimizer, scheduler, 
                           augmentation, num_epochs, patience, device):
    """
    Train with aggressive anti-overfitting techniques:
    - Data augmentation during training
    - L2 regularization
    - Gradient clipping
    - Early stopping based on validation performance
    - Regular validation monitoring
    """
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_mae': [],
        'val_mae': [],
        'train_r2': [],
        'val_r2': [],
        'learning_rates': [],
        'overfitting_ratio': []
    }
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    print("🛡️  STARTING REGULARIZED TRAINING")
    print("=" * 70)
    print(f"{'Epoch':<6} {'Train Loss':<12} {'Val Loss':<12} {'Overfit Ratio':<13} {'Val R²':<10} {'LR':<10}")
    print("=" * 70)
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Training phase with data augmentation
        model.train()
        train_loss = 0.0
        train_mae = 0.0
        num_train_batches = 0
        
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            
            # Apply data augmentation randomly
            if torch.rand(1) < 0.7:  # 70% chance to apply augmentation
                if torch.rand(1) < 0.3:
                    batch_x = augmentation.add_noise(batch_x)
                elif torch.rand(1) < 0.6:
                    batch_x = augmentation.scale_features(batch_x)
                else:
                    batch_x = augmentation.temporal_shift(batch_x)
            
            # Forward pass
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y, model)
            
            # Backward pass with gradient clipping
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()
            
            # Accumulate metrics
            train_loss += loss.item()
            train_mae += torch.mean(torch.abs(outputs - batch_y)).item()
            num_train_batches += 1
        
        # Validation phase (no augmentation)
        model.eval()
        val_loss = 0.0
        val_mae = 0.0
        val_predictions = []
        val_targets = []
        num_val_batches = 0
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                outputs = model(batch_x)
                # Use MSE for validation (without L2 regularization)
                loss = nn.MSELoss()(outputs, batch_y)
                
                val_loss += loss.item()
                val_mae += torch.mean(torch.abs(outputs - batch_y)).item()
                val_predictions.append(outputs.cpu())
                val_targets.append(batch_y.cpu())
                num_val_batches += 1
        
        # Calculate metrics
        avg_train_loss = train_loss / num_train_batches
        avg_val_loss = val_loss / num_val_batches
        avg_train_mae = train_mae / num_train_batches
        avg_val_mae = val_mae / num_val_batches
        
        # Calculate R² for validation
        val_preds = torch.cat(val_predictions, dim=0).numpy()
        val_targs = torch.cat(val_targets, dim=0).numpy()
        
        val_r2_scores = []
        for i in range(val_preds.shape[1]):
            r2 = r2_score(val_targs[:, i], val_preds[:, i])
            val_r2_scores.append(r2)
        avg_val_r2 = np.mean(val_r2_scores)
        
        # Overfitting ratio
        overfitting_ratio = avg_val_loss / avg_train_loss if avg_train_loss > 0 else 1.0
        
        # Store history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_mae'].append(avg_train_mae)
        history['val_mae'].append(avg_val_mae)
        history['val_r2'].append(avg_val_r2)
        history['learning_rates'].append(optimizer.param_groups[0]['lr'])
        history['overfitting_ratio'].append(overfitting_ratio)
        
        # Learning rate scheduling
        scheduler.step(avg_val_loss)
        
        # Print progress
        if epoch % 10 == 0 or epoch < 20:
            print(f"{epoch+1:<6} {avg_train_loss:<12.6f} {avg_val_loss:<12.6f} "
                  f"{overfitting_ratio:<13.3f} {avg_val_r2:<10.4f} {optimizer.param_groups[0]['lr']:<10.2e}")
        
        # Early stopping with improved criteria
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        # Additional early stopping if overfitting gets too severe
        if overfitting_ratio > 3.0 and epoch > 50:
            print(f"\n⚠️  Severe overfitting detected (ratio: {overfitting_ratio:.2f})")
            print("Stopping training early to prevent further overfitting")
            break
            
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered after epoch {epoch+1}")
            print(f"Best validation loss: {best_val_loss:.6f}")
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    training_time = time.time() - start_time
    
    print("=" * 70)
    print(f"Regularized training completed in {training_time:.2f} seconds")
    print(f"Best validation loss: {best_val_loss:.6f}")
    print(f"Final overfitting ratio: {history['overfitting_ratio'][-1]:.3f}")
    print(f"Final validation R²: {history['val_r2'][-1]:.4f}")
    
    return model, history

# Create new DataLoaders with different batch size
reg_train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
reg_val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

reg_train_loader = DataLoader(reg_train_dataset, batch_size=REGULARIZED_BATCH_SIZE, shuffle=True)
reg_val_loader = DataLoader(reg_val_dataset, batch_size=REGULARIZED_BATCH_SIZE, shuffle=False)

# Start regularized training
print("🛡️  TRAINING REGULARIZED MODEL")
print("=" * 60)

regularized_trained_model, regularized_history = train_regularized_model(
    model=regularized_model,
    train_loader=reg_train_loader,
    val_loader=reg_val_loader,
    criterion=regularized_criterion,
    optimizer=regularized_optimizer,
    scheduler=regularized_scheduler,
    augmentation=augmentation,
    num_epochs=REGULARIZED_NUM_EPOCHS,
    patience=REGULARIZED_PATIENCE,
    device=device
)

print("\n✅ Regularized model training completed!")
print("🎯 Anti-overfitting techniques applied successfully!")

🛡️  TRAINING REGULARIZED MODEL
🛡️  STARTING REGULARIZED TRAINING
Epoch  Train Loss   Val Loss     Overfit Ratio Val R²     LR        
1      0.261517     0.010353     0.040         -42.1907   1.00e-04  
1      0.261517     0.010353     0.040         -42.1907   1.00e-04  
2      0.152312     0.004521     0.030         -6.3222    1.00e-04  
2      0.152312     0.004521     0.030         -6.3222    1.00e-04  
3      0.092848     0.004341     0.047         -4.9174    1.00e-04  
3      0.092848     0.004341     0.047         -4.9174    1.00e-04  
4      0.052063     0.004662     0.090         -6.6441    1.00e-04  
4      0.052063     0.004662     0.090         -6.6441    1.00e-04  
5      0.025404     0.004351     0.171         -4.5457    1.00e-04  
5      0.025404     0.004351     0.171         -4.5457    1.00e-04  
6      0.010748     0.004115     0.383         -3.4291    1.00e-04  
6      0.010748     0.004115     0.383         -3.4291    1.00e-04  
7      0.005279     0.004256     0.806

KeyboardInterrupt: 